In [ ]:
### Load libraries and set params
library(Matrix)
library(parallel)
library(Seurat)
library(gridExtra)
library(ggplot2)
library(future)
library(ggpubr)
library(viridis)
library(openxlsx)
library(viridis)
library(scran)
library(gghighlight)
library(dplyr)
library(org.Dm.eg.db)
library(ggExtra)
library(scDblFinder)
library(patchwork)
library(RhpcBLASctl)
blas_set_num_threads(18)
library(peakRAM)
options(device=pdf)
options(future.globals.maxSize = 214748364800)
library(future)
plan("multicore", workers = 18)
library(SeuratDisk)
library(reshape2)
library(tidyverse)
library(RColorBrewer)
library(zellkonverter)

### Set directories
mainDir <- "/data/ebaird/scRNAseq/SCENTINELmar26/"
repDir <- paste0(mainDir, "map_query/")
figDir <- paste0(repDir, "figs/")
tabDir <- paste0(repDir, "tables/")
refsDir <- "/data/ebaird/scRNAseq/refs/"


dir.create(repDir, recursive = TRUE, showWarnings = FALSE)
dir.create(figDir, recursive = TRUE, showWarnings = FALSE)
dir.create(tabDir, recursive = TRUE, showWarnings = FALSE)
dir.create(refsDir, recursive = TRUE, showWarnings = FALSE)

### Set colours
mycols <- c(1, '#ffffe5','#fff7bc','#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506')
mycols11 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray")
mycols13 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray", "blue", "green")
mycols17 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray", "blue", "green", rainbow(4))

mycols20 <- c("yellow", '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "chartreuse", "blue", "green", rainbow(4), "darkslategray3", "darksalmon", "darkorchid4", "cyan", "magenta")

corner <- function(x) x[1:5,1:5]
cols <- c(colorRamps::matlab.like2(20)[1:18], "deeppink2", "deeppink3", "deeppink4")

getdensity <- function(x, y, ...) {
      dens <- MASS::kde2d(x, y, ...)
      ix <- findInterval(x, dens$x)
      iy <- findInterval(y, dens$y)
      ii <- cbind(ix, iy)
      return(dens$z[ii])
}

best_colors <- viridis(20)
best_colors_magma <- magma(20)
best_colors_plasma <- plasma(20)

In [ ]:
# Load seurat objects

seu_query1 <- readRDS("/data/ebaird/scRNAseq/SCENTINELsep24/QC_clustering/merged_clusters.rds")
seu_ref <- readRDS("/data/ebaird/scRNAseq/SCENTINELmar26/QC_clustering/final_clusters.rds")


In [ ]:
# seu_query2 <- readRDS("/data/ebaird/scRNAseq/SCENTINELsep25/QC_clustering/final_clusters.rds")
# seu_query1 <- seu_query2

In [ ]:
seu_ref@meta.data$celltype <- seu_ref@meta.data$seurat_clusters

In [ ]:
# Check active assays
DefaultAssay(seu_ref)
DefaultAssay(seu_query1)

# They should both be "RNA"
DefaultAssay(seu_ref) <- "RNA"
DefaultAssay(seu_query1) <- "RNA"

# How many shared genes?
length(intersect(rownames(seu_ref), rownames(seu_query1)))

seu_ref

In [ ]:
DefaultAssay(seu_ref) <- "SCT"
seu_ref
DefaultAssay(seu_ref) <- "RNA"

In [ ]:
# seu_ref <- NormalizeData(seu_ref, verbose = FALSE)
seu_ref <- FindVariableFeatures(seu_ref, nfeatures = 2716, verbose = FALSE) # match variable features used in reference
# seu_ref <- ScaleData(seu_ref, verbose = FALSE)
# seu_ref <- RunPCA(seu_ref, npcs = 60, verbose = FALSE)

# # Recompute UMAP so it is linked to this PCA. Match the UMAP parameters to those used in the reference, which is what the query will be mapped to.
seu_ref <- RunUMAP(seu_ref, dims = 1:20, reduction = "pca", verbose = FALSE, return.model = TRUE, reduction.name = "umap")
DimPlot(seu_ref, reduction = "umap", group.by = "celltype", label = TRUE, label.size = 3, repel = TRUE) + NoLegend()

In [ ]:
# Normalize
seu_query1 <- NormalizeData(seu_query1, verbose = FALSE)

# IMPORTANT: match reference variable feature count
seu_query1 <- FindVariableFeatures(
  seu_query1,
  selection.method = "vst",
  nfeatures = 2716,
  verbose = FALSE
)

# Scale and PCA
seu_query1 <- ScaleData(seu_query1, verbose = FALSE)
seu_query1 <- RunPCA(seu_query1, npcs = 60, verbose = FALSE)


In [ ]:
anchors <- FindTransferAnchors(
  reference = seu_ref,
  query = seu_query1,
  normalization.method = "LogNormalize",
  reference.reduction = "pca",
  dims = 1:30,
  verbose = TRUE
)


In [ ]:
seu_query1 <- MapQuery(
  anchorset = anchors,
  reference = seu_ref,
  query = seu_query1,
  refdata = list(
    celltype = "celltype"
  ),
  reference.reduction = "pca",
  reduction.model = "umap",
  verbose = TRUE
)


In [ ]:
head(seu_query1[["ref.umap"]]@cell.embeddings)


In [ ]:
seu_query1[["ref.umap"]]@cell.embeddings[, "refUMAP_2"] <- seu_query1[["ref.umap"]]@cell.embeddings[, "refUMAP_2"] * -1

In [ ]:
p1 <- DimPlot(seu_ref, reduction = "umap", group.by = "celltype", label = TRUE, label.size = 3,
    repel = TRUE) + NoLegend() + ggtitle("Reference annotations")
p2 <- DimPlot(seu_query1, reduction = "ref.umap", group.by = "seurat_clusters", label = TRUE,
    label.size = 3, repel = TRUE) + ggtitle("Query transferred labels") + NoLegend()
p1

p2

# Split by genotype
p3 <- DimPlot(seu_query1, reduction = "ref.umap", group.by = "seurat_clusters", label = TRUE,
    label.size = 3, repel = TRUE, split.by = "genotype") + ggtitle("Query transferred labels") 
jpeg(paste0(figDir, "map_query_umap_by_genotype_2024_on_2026.jpg"), width = 16, height = 4, units = "in", res = 300)
p3
dev.off()

In [ ]:
p1 <- DimPlot(seu_ref, reduction = "umap", group.by = "celltype", label = TRUE, label.size = 3,
    repel = TRUE) + NoLegend() + ggtitle("Reference annotations")

clusters_to_show <- c("11")

p2 <- DimPlot(
  seu_query1,
  reduction = "ref.umap",
  group.by = "seurat_clusters",
  cells.highlight = WhichCells(
    seu_query1,
    expression = seurat_clusters %in% clusters_to_show
  ),
  cols.highlight = "red",
  sizes.highlight = 1
) + ggtitle("11")

p1

p2